# Steam Trajectory — Scrape & Database Load

Reads the candidate list frozen by `00_cohort_selection.ipynb`, scrapes SteamCharts for each candidate's monthly history, drops any that fail, freezes the resulting **final cohort** to `cohort_appids.csv` (so downstream notebooks never depend on live network conditions again), and loads everything into `steam_project.db`.

This notebook is standalone: it never re-derives the cohort from the Kaggle metadata, and never needs to reload the full ~126K-game JSON dataset.

In [5]:
%load_ext autoreload
%autoreload 2

import os
os.chdir("/Users/pmacias/Dropbox/steamproject")
print(os.getcwd())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/Users/pmacias/Dropbox/steamproject


In [6]:
import pandas as pd

from steam_trajectory.ingest.steamcharts_scraper import SteamChartsScraper
from steam_trajectory.ingest.kaggle_loader import KaggleLoader
from steam_trajectory.db.connection import get_connection
from steam_trajectory.db.writer import DatabaseWriter

In [7]:
candidates = pd.read_csv("candidate_appids.csv")
print(f"Loaded {len(candidates)} candidates")
candidates.head()

Loaded 260 candidates


,AppID,Name,Release date,Developers,Publishers,Price,total_reviews,review_score_percent,Genres
0,1030210,Torchlight III,"Oct 13, 2020",Echtra Inc.,Arc Games,5.99,11230,46.527159,"Action, Adventure, RPG"
1,852090,Penko Park,"Oct 23, 2020",Ghostbutter,Ghostbutter,3.49,578,98.961938,"Adventure, Indie, Simulation"
2,1100420,Praetorians - HD Remaster,"Jan 24, 2020","Torus Games, Pyro Studios",Kalypso Media,7.99,985,91.370558,Strategy
3,1248130,Farming Simulator 22,"Nov 21, 2021",Giants Software,Giants Software,19.99,84512,92.424744,Simulation
4,1808780,祖玛少女/Zuma Girls,"May 1, 2022",Twilight Sonata Studio,Twilight Sonata Studio,1.19,684,82.163743,Casual


## Quick scraper sanity check
Optional — a fast check that the scraper still works before committing to the full batch. Safe to skip on reruns once you trust it.

In [8]:
scraper = SteamChartsScraper()
test_appids = candidates["AppID"].head(3).tolist()
print("Testing:", test_appids)

for appid in test_appids:
    history = scraper.get_monthly_history(appid)
    print(f"\nAppID {appid}: {len(history)} months of data")
    print(history[:3])

Testing: [1030210, 852090, 1100420]

AppID 1030210: 73 months of data
[{'appid': 1030210, 'month': '2026-06-01', 'avg_players': 22.82, 'peak_players': 45, 'est_owners_low': None, 'est_owners_high': None, 'source': 'steamcharts'}, {'appid': 1030210, 'month': '2026-05-01', 'avg_players': 21.42, 'peak_players': 50, 'est_owners_low': None, 'est_owners_high': None, 'source': 'steamcharts'}, {'appid': 1030210, 'month': '2026-04-01', 'avg_players': 18.74, 'peak_players': 38, 'est_owners_low': None, 'est_owners_high': None, 'source': 'steamcharts'}]

AppID 852090: 31 months of data
[{'appid': 852090, 'month': '2025-07-01', 'avg_players': 1.96, 'peak_players': 11, 'est_owners_low': None, 'est_owners_high': None, 'source': 'steamcharts'}, {'appid': 852090, 'month': '2024-12-01', 'avg_players': 1.64, 'peak_players': 7, 'est_owners_low': None, 'est_owners_high': None, 'source': 'steamcharts'}, {'appid': 852090, 'month': '2024-11-01', 'avg_players': 2.25, 'peak_players': 10, 'est_owners_low': None,

## Full scrape + freeze final cohort
Scrapes every candidate, drops any that fail (no SteamCharts page, timeout, etc. — see notes in `steamcharts_scraper.py`), and **freezes the resulting cohort to `cohort_appids.csv`**. This is the fix for the reproducibility gap we hit earlier: since scraping failures depend partly on live network/server conditions, rerunning this cell can legitimately produce a different-sized survivor set each time. Freezing the result means every notebook from here on reads a fixed list, rather than silently re-deriving a possibly-different cohort on every rerun.

In [9]:
all_appids = candidates["AppID"].tolist()
all_histories, failures = scraper.get_monthly_history_batch(all_appids)

print(f"\nDone. {len(all_histories)} monthly records, {len(failures)} candidates failed.")
print(failures)

Processed 20/260 games (4 failures so far)...
Processed 40/260 games (9 failures so far)...
Processed 60/260 games (12 failures so far)...
Processed 80/260 games (15 failures so far)...
Processed 100/260 games (17 failures so far)...
Processed 120/260 games (20 failures so far)...
Processed 140/260 games (22 failures so far)...
Processed 160/260 games (27 failures so far)...
Processed 180/260 games (32 failures so far)...
Processed 200/260 games (35 failures so far)...
Processed 220/260 games (39 failures so far)...
Processed 240/260 games (42 failures so far)...
Processed 260/260 games (45 failures so far)...

Done. 13594 monthly records, 45 candidates failed.
[{'appid': 1808780, 'error': '500 Server Error: Internal Server Error for url: https://steamcharts.com/app/1808780'}, {'appid': 1043390, 'error': '500 Server Error: Internal Server Error for url: https://steamcharts.com/app/1043390'}, {'appid': 1345740, 'error': '500 Server Error: Internal Server Error for url: https://steamchar

In [10]:
failed_appids = {f["appid"] for f in failures}
final_cohort = candidates[~candidates["AppID"].isin(failed_appids)].reset_index(drop=True)

final_appids = set(final_cohort["AppID"])
final_histories = [record for record in all_histories if record["appid"] in final_appids]

print(f"Final cohort size: {len(final_cohort)}")
print(f"Monthly records for final cohort: {len(final_histories)}")

final_cohort.to_csv("cohort_appids.csv", index=False)
print("Saved final cohort to cohort_appids.csv")

Final cohort size: 215
Monthly records for final cohort: 13594
Saved final cohort to cohort_appids.csv


## Load into the database
`KaggleLoader.iter_games` / `iter_genres` are called as staticmethods here — no `KaggleLoader` instance needed, so this never reloads the full JSON dataset.

In [11]:
conn = get_connection("steam_project.db")
writer = DatabaseWriter(conn)

# Clear out any previously-loaded monthly metrics before reloading —
# makes this cell safe to rerun from scratch rather than accumulating
# duplicate/stale rows on repeated runs.
conn.execute("DELETE FROM monthly_metrics")
conn.commit()

# 1. Games
for game in KaggleLoader.iter_games(final_cohort):
    writer.insert_game(**game)

# 2. Genre links
for appid, genre_name in KaggleLoader.iter_genres(final_cohort):
    writer.link_game_genre(appid, genre_name)

# 3. Monthly metrics from the scrape
for record in final_histories:
    writer.insert_monthly_metric(**record)

writer.commit()
print("Database load complete.")

Database load complete.


## Verify the load

In [12]:
print("games:", pd.read_sql("SELECT COUNT(*) as n FROM games", conn)["n"][0])
print("game_genres:", pd.read_sql("SELECT COUNT(*) as n FROM game_genres", conn)["n"][0])
print("monthly_metrics:", pd.read_sql("SELECT COUNT(*) as n FROM monthly_metrics", conn)["n"][0])

pd.read_sql("""
    SELECT g.name, g.release_date, gen.name AS genre, COUNT(m.month) AS months_tracked
    FROM games g
    JOIN game_genres gg ON g.appid = gg.appid
    JOIN genres gen ON gg.genre_id = gen.genre_id
    JOIN monthly_metrics m ON g.appid = m.appid
    GROUP BY g.appid, gen.name
    LIMIT 10
""", conn)

games: 215
game_genres: 589
monthly_metrics: 13594


,name,release_date,genre,months_tracked
0,GearCity,"Jan 14, 2022",Indie,145
1,GearCity,"Jan 14, 2022",Simulation,145
2,GearCity,"Jan 14, 2022",Strategy,145
3,ShellShock Live,"May 22, 2020",Action,136
4,ShellShock Live,"May 22, 2020",Casual,136
5,ShellShock Live,"May 22, 2020",Indie,136
6,ShellShock Live,"May 22, 2020",Massively Multiplayer,136
7,ShellShock Live,"May 22, 2020",Strategy,136
8,Tales from the Borderlands,"Feb 16, 2021",Adventure,139
9,Infernax,"Feb 14, 2022",Action,53


In [15]:
frozen_appids = set(pd.read_csv("cohort_appids.csv")["AppID"])
db_appids = set(pd.read_sql("SELECT appid FROM games", conn)["appid"])
print("Match:",  frozen_appids == db_appids)


Match: True
